# 01 — Exploratory Data Analysis
## Tamil Nadu Assembly Elections (1971–2021)

**Objectives:**
- Data quality audit (nulls, types, duplicates)
- Univariate distributions (votes, margin, turnout, age, n_cand)
- Temporal overview (general elections vs by-elections, turnout trends)
- Categorical distributions (sex, education, profession, constituency_type)

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from db import query, query_all, GENERAL_ELECTION_YEARS, POST_DELIM_YEARS, MAJOR_PARTIES

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [ ]:
# Load all data
df = query_all()
print(f'Total records: {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(f'Years: {sorted(df.year.unique())}')
df.head()

## 1. Data Quality Audit

In [ ]:
# Null counts per column
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_df = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_df = null_df[null_df.null_count > 0].sort_values('null_pct', ascending=False)
print(f'Columns with nulls: {len(null_df)} / {len(df.columns)}')
null_df

In [ ]:
# Data types
print('Data types:')
print(df.dtypes.value_counts())
print()
# Check for duplicates
dupes = df.duplicated(subset=['year', 'constituency_no', 'candidate']).sum()
print(f'Duplicate (year, constituency, candidate) rows: {dupes}')

In [ ]:
# Visualize null pattern
fig, ax = plt.subplots(figsize=(14, 8))
null_matrix = df.isnull().astype(int)
# Sample for visibility
sample_idx = np.linspace(0, len(df)-1, 500, dtype=int)
sns.heatmap(null_matrix.iloc[sample_idx].T, cbar=False, yticklabels=True, cmap='YlOrRd', ax=ax)
ax.set_title('Null Pattern Across Columns (500-row sample)', fontsize=14)
ax.set_xlabel('Record (sampled)')
plt.tight_layout()
plt.savefig('/app/output/01_null_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Univariate Distributions

In [ ]:
# Filter to general elections for distribution analysis
ge = df[df.year.isin(GENERAL_ELECTION_YEARS)].copy()
print(f'General election records: {len(ge):,} / {len(df):,} total')

In [ ]:
# Key numeric distributions
numeric_cols = ['votes', 'margin', 'turnout_percentage', 'vote_share_percentage', 'age', 'n_cand', 'enop']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data = ge[col].dropna()
    axes[i].hist(data, bins=50, edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col}\nμ={data.mean():.1f}, σ={data.std():.1f}', fontsize=11)
    axes[i].axvline(data.median(), color='red', linestyle='--', alpha=0.7, label=f'median={data.median():.1f}')
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Numeric Distributions (General Elections Only)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/app/output/01_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Winner vote share distribution by election year
winners = ge[ge.position == 1].copy()
fig = px.box(winners, x='year', y='vote_share_percentage', 
             title='Winner Vote Share Distribution by Election Year',
             labels={'year': 'Election Year', 'vote_share_percentage': 'Vote Share (%)'})
fig.update_layout(height=500)
fig.show()

## 3. Temporal Overview

In [ ]:
# Records per year — distinguish general vs by-elections
year_counts = df.groupby('year').agg(
    candidates=('candidate', 'count'),
    constituencies=('constituency_no', 'nunique'),
    election_type=('election_type', 'first')
).reset_index()
year_counts['is_general'] = year_counts.year.isin(GENERAL_ELECTION_YEARS)

fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#2196F3' if g else '#FFC107' for g in year_counts.is_general]
bars = ax.bar(year_counts.year, year_counts.candidates, color=colors, edgecolor='white')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Candidates')
ax.set_title('Candidates per Election Year (Blue=General, Yellow=By-election)')

for bar, count in zip(bars, year_counts.constituencies):
    if bar.get_height() > 100:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                f'{count}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('/app/output/01_candidates_per_year.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Turnout trend over general elections
turnout_by_year = ge.groupby('year').agg(
    avg_turnout=('turnout_percentage', 'mean'),
    median_turnout=('turnout_percentage', 'median'),
    total_electors=('electors', 'sum'),
    total_valid_votes=('valid_votes', lambda x: x.drop_duplicates().sum())
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(turnout_by_year.year, turnout_by_year.avg_turnout, 'o-', color='#2196F3', linewidth=2, markersize=8, label='Avg Turnout %')
ax1.fill_between(turnout_by_year.year, turnout_by_year.avg_turnout, alpha=0.1, color='#2196F3')
ax1.set_xlabel('Election Year')
ax1.set_ylabel('Average Turnout (%)', color='#2196F3')
ax1.set_title('Voter Turnout Trend — Tamil Nadu General Elections')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

for _, row in turnout_by_year.iterrows():
    ax1.annotate(f"{row.avg_turnout:.1f}%", (row.year, row.avg_turnout),
                textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/app/output/01_turnout_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Average candidates per constituency over time
cands_per_const = ge.groupby('year').agg(
    avg_candidates=('n_cand', 'mean'),
    total_candidates=('candidate', 'count'),
    constituencies=('constituency_no', 'nunique')
).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(cands_per_const.year, cands_per_const.avg_candidates, color='#FF7043', edgecolor='white')
ax.set_xlabel('Election Year')
ax.set_ylabel('Average Candidates per Constituency')
ax.set_title('Electoral Competition Intensity Over Time')
for _, row in cands_per_const.iterrows():
    ax.annotate(f"{row.avg_candidates:.1f}", (row.year, row.avg_candidates),
                textcoords='offset points', xytext=(0, 5), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/app/output/01_competition_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Categorical Distributions

In [ ]:
# Gender distribution over time
gender_by_year = ge.groupby(['year', 'sex']).size().unstack(fill_value=0)
gender_pct = gender_by_year.div(gender_by_year.sum(axis=1), axis=0) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Count
gender_by_year.plot(kind='bar', stacked=True, ax=ax1, color=['#E91E63', '#2196F3', '#9E9E9E'])
ax1.set_title('Candidate Count by Gender per Election')
ax1.set_xlabel('Election Year')
ax1.set_ylabel('Count')
ax1.legend(title='Sex')

# Female percentage trend
if 'F' in gender_pct.columns:
    ax2.plot(gender_pct.index, gender_pct['F'], 'o-', color='#E91E63', linewidth=2, markersize=8)
    ax2.fill_between(gender_pct.index, gender_pct['F'], alpha=0.1, color='#E91E63')
    ax2.set_title('Female Candidate Percentage Over Time')
    ax2.set_xlabel('Election Year')
    ax2.set_ylabel('Female %')
    for yr, pct in zip(gender_pct.index, gender_pct['F']):
        ax2.annotate(f'{pct:.1f}%', (yr, pct), textcoords='offset points', xytext=(0, 8), ha='center')

plt.tight_layout()
plt.savefig('/app/output/01_gender_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Education distribution (recent elections only — data availability)
recent = ge[ge.year >= 2011].copy()
edu_counts = recent.myneta_education.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
edu_counts.plot(kind='barh', ax=ax, color='#7E57C2', edgecolor='white')
ax.set_title('Education Level Distribution (2011–2021 General Elections)')
ax.set_xlabel('Count')
ax.set_ylabel('')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/app/output/01_education_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Profession distribution (recent elections)
prof_counts = recent.tcpd_prof_main.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
prof_counts.plot(kind='barh', ax=ax, color='#26A69A', edgecolor='white')
ax.set_title('Primary Profession Distribution (2011–2021 General Elections)')
ax.set_xlabel('Count')
ax.set_ylabel('')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/app/output/01_profession_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Constituency type distribution
const_type = ge.drop_duplicates(subset=['year', 'constituency_no']).groupby(['year', 'constituency_type']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
const_type.plot(kind='bar', stacked=True, ax=ax, color=['#FF7043', '#42A5F5', '#66BB6A'])
ax.set_title('Constituency Type Distribution per Election')
ax.set_xlabel('Election Year')
ax.set_ylabel('Number of Constituencies')
ax.legend(title='Type')
plt.tight_layout()
plt.savefig('/app/output/01_constituency_types.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
print('=== EDA SUMMARY ===')
print(f'Total records: {len(df):,}')
print(f'General election records: {len(ge):,}')
print(f'Years covered: {df.year.min()}–{df.year.max()} ({df.year.nunique()} unique)')
print(f'General elections: {len(GENERAL_ELECTION_YEARS)}')
print(f'Parties: {df.party.nunique()}')
print(f'Constituencies: {df.constituency_no.nunique()}')
print(f'Districts (post-2008): {ge[ge.year >= 2011].district_name.nunique()}')
print(f'\nAvg turnout: {ge.turnout_percentage.mean():.1f}%')
print(f'Avg candidates/constituency: {ge.n_cand.mean():.1f}')
print(f'Female candidates (recent): {(recent.sex == "F").mean()*100:.1f}%')
print(f'Records with null district: {df.district_name.isnull().sum():,} (pre-delimitation)')